# 06 - Quantitative Backtesting: Momentum Breakout Strategy

Event-driven backtesting of the Momentum Breakout strategy with execution slippage modeling (Corwin-Schultz spread proxy and square-root market impact).


In [ ]:
import asyncio
from datetime import timedelta

from httpx import ASGITransport, AsyncClient

from apps.api.main import create_app
from src.utils.time import utc_now

print('Backtesting API client ready.')


In [ ]:
async def run_sample_backtest():
    app = create_app()
    transport = ASGITransport(app=app)
    headers = {"X-API-Key": "dev-api-key-researcher-1"}
    async with AsyncClient(transport=transport, base_url="http://test") as client:
        payload = {
            "strategy_name": "MomentumBreakout",
            "symbols": ["SOL", "NEAR", "RENDER"],
            "start_date": (utc_now() - timedelta(days=90)).isoformat(),
            "end_date": utc_now().isoformat(),
            "initial_capital": 100000.0,
            "maker_fee_bps": 2.0,
            "taker_fee_bps": 5.0,
            "slippage_model": "market_impact",
        }
        resp = await client.post("/api/v1/backtests/run", json=payload, headers=headers)
        assert resp.status_code == 200, f"Error: {resp.text}"
        data = resp.json()
        print('Backtest execution completed via API.')
        print(f"Run ID: {data['id']}")
        print(f"Strategy: {data['strategy_name']}")
        print(f"Total Return: {data['total_return_pct']:+.2f}%")
        print(f"Sharpe Ratio: {data['sharpe_ratio']:.2f}")
        print(f"Max Drawdown: {data['max_drawdown_pct']:.2f}%")
        print(f"Total Trades: {len(data.get('trades') or [])}")

asyncio.run(run_sample_backtest())
